# Here one can find the code for my visualizations

In [1]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
import re

# Let's load our Data

In [2]:
df_kNN_FastText_Warriner = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_FastText_kNN_Warriner.csv'))
df_LinReg_FastText_Warriner = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_FastText_LinReg_Warriner.csv'))
df_LinReg_SGNS_Warriner = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_LinReg_Warriner.csv'))
df_LinReg_FastText_NRC_VAD = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_FastText_LinReg_NRC_VAD.csv'))
df_LinReg_SGNS_NRC_VAD = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_LinReg_NRC_VAD.csv'))
df_PaRaSim_SGNS_NRC_VAD = pd.read_csv(Path('HistoricalVAD/Correlations/Summary_SGNS_PaRaSim_NRC_VAD.csv'))

# Means Means Means

# Warriner

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# One entry per model: dataframe, display name, marker color/symbol.
# Colors: "Tableau classic" palette (orange / red / blue) - good contrast
# against white, colorblind-checked. Distinct marker shapes (diamond /
# square / circle) keep series distinguishable even where points overlap.
# Order matters here: traces are drawn in list order, later ones on top.
# The diamond (LinReg FastText) is listed last so it doesn't get fully
# covered by the circle (LinReg SGNS), and gets a dark edge + slightly
# larger size so it stays visible even where points overlap.
models = [
    {"df": df_LinReg_FastText_Warriner, "name": "LinReg_FastText", "color": "white" , "symbol": "circle", "size": 8.5, "edge": "black"},
    {"df": df_kNN_FastText_Warriner, "name": "kNN_FastText", "color": "navy" , "symbol": "square", "size": 6, "edge": "navy"},
    {"df": df_LinReg_SGNS_Warriner, "name": "LinReg_SGNS", "color": "dodgerblue", "symbol": "diamond", "size": 6, "edge": "white"}
]

# One entry per subplot: dataframe column and (row, col) position.
dimensions = [
    {"col": "r_mean", "title": "Mean Development", "row": 1, "col_pos": 1},
    {"col": "r_V", "title": "Valence Development", "row": 1, "col_pos": 2},
    {"col": "r_A", "title": "Arousal Development", "row": 2, "col_pos": 1},
    {"col": "r_D", "title": "Dominance Development", "row": 2, "col_pos": 2},
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[d["title"] for d in dimensions],
    vertical_spacing=0.15,
)

x_values = [i * 1000 for i in range(2, 9)]

for dim in dimensions:
    for i, model in enumerate(models):
        y_values = model["df"][dim["col"]]
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values,
                name=model["name"],
                mode='markers',
                marker=dict(
                    color=model["color"],
                    symbol=model["symbol"],
                    size=model["size"],
                    opacity=1.0,
                    line=dict(width=1.5, color=model["edge"]),
                ),
                showlegend=(dim["row"] == 1 and dim["col_pos"] == 1),
            ),
            row=dim["row"], col=dim["col_pos"]
        )

# Update all axes
"""
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text="Lexicon Size in Words", row=i, col=j)
        fig.update_yaxes(title_text="Pearson's r", range=[0.1, 0.58], row=i, col=j)
"""

fig.update_annotations(font_size=14)
fig.update_layout(
    title=dict(
        text="Warriner Performance Development",
        x=0.5, xanchor='center',
        y=0.95, yanchor='top',
        font=dict(size=18),
    ),
    height=500,
    width=1000,
    margin=dict(l=80, r=80, t=100, b=40),
    font=dict(size=12),
    title_font=dict(size=14),
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
        entrywidth=120, entrywidthmode='pixels',
        tracegroupgap=20,
    )
)
 
fig.write_image("warriner_lexsize.svg")
fig.show()

# NRC-VAD

In [8]:
current_index = df_LinReg_FastText_NRC_VAD.index.tolist()
new_index_order = current_index[1:] + current_index[:1]
df_LinReg_FastText_NRC_VAD = df_LinReg_FastText_NRC_VAD.reindex(new_index_order)
df_LinReg_SGNS_NRC_VAD = df_LinReg_SGNS_NRC_VAD.reindex(new_index_order)
df_PaRaSim_SGNS_NRC_VAD = df_PaRaSim_SGNS_NRC_VAD.reindex(new_index_order)

In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# One entry per model: dataframe, display name, marker color/symbol.
# Colors: "Tableau classic" palette (orange / red / blue) - good contrast
# against white, colorblind-checked. Distinct marker shapes (diamond /
# square / circle) keep series distinguishable even where points overlap.
# Order matters here: traces are drawn in list order, later ones on top.
# The diamond (LinReg FastText) is listed last so it doesn't get fully
# covered by the circle (LinReg SGNS), and gets a dark edge + slightly
# larger size so it stays visible even where points overlap.
models = [
    {"df": df_LinReg_FastText_NRC_VAD, "name": "LinReg_FastText", "color": "white" , "symbol": "circle", "size": 8.5, "edge": "black"},
    {"df": df_LinReg_SGNS_NRC_VAD, "name": "LinReg_SGNS", "color": "dodgerblue" , "symbol": "square", "size": 6, "edge": "white"},
    {"df": df_PaRaSim_SGNS_NRC_VAD, "name": "PaRaSim_SGNS", "color": "navy", "symbol": "diamond", "size": 6, "edge": "navy"}
]

# One entry per subplot: dataframe column and (row, col) position.
dimensions = [
    {"col": "r_mean", "title": "Mean Development", "row": 1, "col_pos": 1},
    {"col": "r_V", "title": "Valence Development", "row": 1, "col_pos": 2},
    {"col": "r_A", "title": "Arousal Development", "row": 2, "col_pos": 1},
    {"col": "r_D", "title": "Dominance Development", "row": 2, "col_pos": 2},
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[d["title"] for d in dimensions],
    vertical_spacing=0.15,
)

x_values = [i * 1000 for i in range(2, 11)]

for dim in dimensions:
    for i, model in enumerate(models):
        y_values = model["df"][dim["col"]]
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=y_values,
                name=model["name"],
                mode='markers',
                marker=dict(
                    color=model["color"],
                    symbol=model["symbol"],
                    size=model["size"],
                    opacity=1.0,
                    line=dict(width=1.5, color=model["edge"]),
                ),
                showlegend=(dim["row"] == 1 and dim["col_pos"] == 1),
            ),
            row=dim["row"], col=dim["col_pos"]
        )

# Update all axes
"""
for i in range(1, 3):
    for j in range(1, 3):
        fig.update_xaxes(title_text="Lexicon Size in Words", row=i, col=j)
        fig.update_yaxes(title_text="Pearson's r", range=[0.07, 0.6], row=i, col=j)
"""

fig.update_annotations(font_size=14)
fig.update_layout(
    title=dict(
        text="NRC-VAD Performance Development",
        x=0.5, xanchor='center',
        y=0.95, yanchor='top',
        font=dict(size=18),
    ),
    height=500,
    width=1000,
    margin=dict(l=80, r=80, t=100, b=40),
    font=dict(size=12),
    title_font=dict(size=14),
    showlegend=True,
    legend=dict(
        orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5,
        entrywidth=120, entrywidthmode='pixels',
        tracegroupgap=20,
    )
)
 
fig.write_image("nrcvad_lexsize.svg")
fig.show()

# Visualization of Variance

In [14]:
liste_lexicon_names = ['FastText_kNN_Warriner', 'FastText_LinearRegression_Warriner', 'SGNS_LinearRegression_Warriner', 'FastText_LinearRegression_NRC_VAD', 'SGNS_LinearRegression_NRC_VAD', 'SGNS_PaRaSim_NRC_VAD']

In [19]:
def vis(lexicon_name, title):
    lexicon_folder = []
    folder = Path('HistoricalVAD/Correlations/')
    lexicon_folder.extend(folder.glob(f'{lexicon_name}[0-9]*'))
    
    numbers = []
    data = []
    
    for path in lexicon_folder:
        match = re.search(r'\d+', str(path))
        n = int(match.group())
        numbers.extend([n]*50)
        
        df = pd.read_csv(path)
        data.extend(list(df['single_mean'])[:-1])

    listA = []
    listA2 = []
    listB = []
    listB2 = []
    listC = []
    listC2 = []
    listD = []
    listD2 = []
    listE = []
    listE2 = []

    for i in range(len(data)):
        if data[i] <= 0.25:
            listA.append(data[i])
            listA2.append(numbers[i])
        elif data[i] <=0.3:
            listB.append(data[i])
            listB2.append(numbers[i])
        elif data[i] <=0.35:
            listC.append(data[i])
            listC2.append(numbers[i])
        elif data[i] <= 0.4:
            listD.append(data[i])
            listD2.append(numbers[i])
        else:
            listE.append(data[i])
            listE2.append(numbers[i])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=listA2, y= listA, name='<= 0.25',
                            mode = 'markers',  # Show only markers (dots)
                            marker = dict(color='dodgerblue', size=15, opacity=0.7,)))
    fig.add_trace(go.Scatter(x=listB2, y= listB, name='<= 0.3',
                            mode = 'markers',  # Show only markers (dots)
                            marker = dict(color='skyblue', size=15, opacity=0.9,)))
    fig.add_trace(go.Scatter(x=listC2, y= listC, name='<= 0.35',
                            mode = 'markers',  # Show only markers (dots)
                            marker = dict(color='lightblue', size=15, opacity=0.9,)))
    fig.add_trace(go.Scatter(x=listD2, y= listD, name='<= 0.4',
                            mode = 'markers',  # Show only markers (dots)
                            marker = dict(color='steelblue', size=15, opacity=0.7,)))
    fig.add_trace(go.Scatter(x=listE2, y= listE, name='> 0.4',
                            mode = 'markers',  # Show only markers (dots)
                            marker = dict(color='navy', size=15, opacity=0.6,)))
    fig.update_layout(
        title={
            'text': "<b>Mean Distribution of Sub-Models</b><br><span style='font-size:14px; color:gray'> "+title+"</span>",
            'x': 0.5,
            'xanchor': 'center',
            'y': 0.90,
            'font': {
                'size': 24,
                'family': "Arial",
                'color': "black"
            },
            'yanchor': 'top'  # This ensures proper alignment with subtitle
        },
        xaxis_title="Lexicon Size in Words",
        yaxis_title="Pearson's r",
        yaxis=dict(
            range=[0.15, 0.47],  # Fixed y-axis scale
            autorange=False  # Disable auto-scaling (optional)
        )
    )

    # Save each call's plot as its own SVG, named after the lexicon/model it covers.
    # width/height set explicitly so the exported file has a wide, non-tall aspect
    # ratio when scaled to \linewidth in LaTeX (independent of fig.show()'s sizing).
    fig.write_image(f"variation_{lexicon_name}.svg", width=1000, height=500)

    fig.show()

In [17]:
liste_lexicon_names

['FastText_kNN_Warriner',
 'FastText_LinearRegression_Warriner',
 'SGNS_LinearRegression_Warriner',
 'FastText_LinearRegression_NRC_VAD',
 'SGNS_LinearRegression_NRC_VAD',
 'SGNS_PaRaSim_NRC_VAD']

In [20]:
vis('FastText_kNN_Warriner', 'FastText_kNN (Warriner)')
vis('FastText_LinearRegression_Warriner', 'FastText_LinReg (Warriner)')
vis('SGNS_LinearRegression_Warriner', 'SGNS_LinReg (Warriner)')
vis('FastText_LinearRegression_NRC_VAD', 'FastText_LinReg (NRC-VAD)')
vis('SGNS_LinearRegression_NRC_VAD', 'SGNS_LinReg (NRC-VAD)')
vis('SGNS_PaRaSim_NRC_VAD', 'PaRaSim_SGNS (NRC-VAD)')